# SILO and User-Supplied Daily Rainfall

This notebook demonstrates how to bypass the internal Fortran daily rainfall generator by using daily rainfall data from external sources. We will specifically look at:
1. Fetching daily data from the **SILO Point Data API**.
2. Preparing that data for `pyraingen`.
3. Injecting the data into the sub-daily disaggregation process.

This approach is highly recommended as it removes the need for a Fortran compiler and allows you to use high-quality, gauge-based or gridded daily data directly in your workflow.

## 1. Fetching BOM Gauge Data via SILO

While the Bureau of Meteorology (BOM) provides data via its website, it is difficult to access programmatically. The **SILO Point Data API** (Long Paddock) is the standard alternative for researchers. It ingests BOM gauge data, cleans/patches it, and provides a simple API. You can fetch data for a specific **BOM Station Number** (e.g., `040075` for Esk Post Office) or by coordinates.

In [ ]:
from pyraingen.silo import get_silo_point_data, prepare_silo_for_pyraingen

# Parameters for SILO using a BOM Station Number
site = "040075"       # BOM Station Number (Esk Post Office, QLD)
start_date = "19800101"
end_date = "20231231"
email = "your.email@example.com"  # Required by SILO API

# Fetch the data as a Pandas DataFrame
try:
    silo_df = get_silo_point_data(site, start_date, end_date, email)
    print(f"Successfully fetched SILO data for BOM station {site}!")
    print(silo_df.head())
except Exception as e:
    print(f"Error fetching data: {e}")
    print("Note: You must provide a valid email address and have an internet connection.")

## 2. Preparing Data for pyraingen

The `pyraingen` sub-daily process expects a NumPy array of daily rainfall. We can use the helper function `prepare_silo_for_pyraingen` to extract the correct series and ensure it matches the requested simulation years.

In [ ]:
sim_year_start = 2000
sim_year_end = 2010

# Extract the rainfall array
# In a real scenario, you would have the silo_df from the step above
if 'silo_df' in locals():
    daily_rain = prepare_silo_for_pyraingen(silo_df, sim_year_start, sim_year_end)
    print(f"Daily rainfall array shape: {daily_rain.shape}")
    print(f"First 5 days: {daily_rain[:5]}")

## 3. Running Sub-daily Disaggregation

Now we can run `regionalisedsubdailysim` by passing our `daily_rain` array to the `suppliedDailyRain` parameter. We set `genSeqOption=5` to tell the package to use this supplied data instead of looking for a NetCDF file.

In [ ]:
from pyraingen.regionalisedsubdailysim import regionalisedsubdailysim
import os

# Paths for fragments and indices (you'll need to point these to your actual data folders)
pathSubDaily = "path/to/sub_daily_data/"
targetIndex = 66062

if 'daily_rain' in locals():
    # Note: ensure pathSubDaily points to a valid directory containing the pluviograph NetCDF files
    # regionalisedsubdailysim(
    #     fnameInput=None,             # Not used when suppliedDailyRain is provided
    #     pathSubDaily=pathSubDaily,
    #     targetIndex=targetIndex,
    #     suppliedDailyRain=daily_rain,
    #     genSeqOption=5,              # Option for user-supplied data
    #     nSims=1,                     # Number of simulations to perform
    #     fnameSubDaily="silo_subdaily.nc"
    # )

## 4. Using BOM Gauge Data or AWAP

If you have daily rainfall data from the Bureau of Meteorology (BOM) in a CSV format or from the AWAP gridded dataset, you can follow a similar pattern:

1. Load your data into a NumPy array.
2. Ensure the array represents a continuous sequence of days.
3. Pass it to `suppliedDailyRain`.

```python
import pandas as pd
import numpy as np

# Example: Load a BOM CSV file
# df = pd.read_csv("BOM_data_station_XXXX.csv")
# daily_values = df['Rainfall amount (millimetres)'].values

# Inject into pyraingen
# regionalisedsubdailysim(..., suppliedDailyRain=daily_values, genSeqOption=5)
```